# Pre-Training on SlimPajama-6B (Azure)

**Model**: Decoder-only Transformer with GQA + REPO-Attention + Flash-Attention  
**Dataset**: [SlimPajama-6B](https://hub.oxen.ai/datasets/SlimPajama-6B) via Oxen  
**Tracking**: Weights & Biases  

### Dataset Schema (from Oxen)
| Column | Type | Description |
|--------|------|-------------|
| `text`  | str | Raw document text |
| `meta`  | struct | Contains `redpajama_set_name` (C4, CommonCrawl, StackExchange, etc.) |
| `__index_level_0__` | int | Row index |

Uses existing code from `train/` folder: `tokenizer.py`, `dataset_define.py`, `save_checkpoint.py`, and `transformer/build_transformer.py`.

## 0. Install Dependencies (run once on Azure)

In [ ]:
# Uncomment and run on Azure VM if packages are missing
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# !pip install oxen wandb transformers tokenizers pyarrow

## 1. Setup Paths & Imports

In [ ]:
import os
import sys
import time
import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader
from datetime import datetime
import wandb

# ---- Set PROJECT_ROOT to the Transformers folder ----
PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__"))
TRAIN_DIR = os.path.join(PROJECT_ROOT, "train")

# Add both to sys.path so we can import our existing modules
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, TRAIN_DIR)

# ---- Import YOUR existing code ----
# transformer/ is a proper package with __init__.py
from transformer.build_transformer import build_transformer

# train/ files are imported directly (TRAIN_DIR is on sys.path)
from dataset_define import SlimPajamaDataset
from save_checkpoint import save_checkpoint
from tokenizer import tokenizer

print(f"Project root  : {PROJECT_ROOT}")
print(f"Train dir     : {TRAIN_DIR}")
print(f"PyTorch       : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
# ======================== PATHS ========================
DATASET_DIR    = os.path.join(PROJECT_ROOT, "SlimPajama-6B")  # where oxen clones to
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")

# ======================== MODEL ========================
D_MODEL    = 768
NUM_LAYERS = 12
NUM_HEADS  = 12
KV_HEADS   = 4
D_FF       = 3072
DROPOUT    = 0.1
MAX_SEQ_LEN = 2048

USE_REPO  = True    # REPO-Attention (learned positions)
USE_FLASH = True    # Flash-Attention (PyTorch >= 2.0)

# ======================== TRAINING ========================
EPOCHS         = 3
BATCH_SIZE     = 8
LEARNING_RATE  = 3e-4
WEIGHT_DECAY   = 0.01
MAX_GRAD_NORM  = 1.0
WARMUP_STEPS   = 500

# ======================== WANDB ========================
WANDB_PROJECT = "Spedrox_llm"
USE_WANDB     = True

# ======================== DEVICE ========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tokenizer info
VOCAB_SIZE = len(tokenizer)
PAD_TOKEN_ID = tokenizer.pad_token_id

print(f"Vocab size    : {VOCAB_SIZE}")
print(f"Pad token ID  : {PAD_TOKEN_ID}")
print(f"Device        : {device}")
print(f"Dataset dir   : {DATASET_DIR}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")

## 3. Clone SlimPajama-6B via Oxen

In [ ]:
import oxen

if not os.path.exists(DATASET_DIR):
    print("Cloning SlimPajama-6B dataset via Oxen...")
    oxen.clone("https://hub.oxen.ai/datasets/SlimPajama-6B", DATASET_DIR)
    print("Clone complete!")
else:
    print(f"Dataset already exists at {DATASET_DIR}")

# List what we got
for root, dirs, files in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, "").count(os.sep)
    if level < 2:
        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/")
        for f in files[:10]:
            print(f"{indent}  {f}")
        if len(files) > 10:
            print(f"{indent}  ... and {len(files)-10} more files")

## 4. Build Model (using your `build_transformer`)

In [ ]:
model = build_transformer(
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    src_seq_len=MAX_SEQ_LEN,
    tgt_seq_len=MAX_SEQ_LEN,
    d_model=D_MODEL,
    N=NUM_LAYERS,
    h=NUM_HEADS,
    kv_h=KV_HEADS,
    dropout=DROPOUT,
    d_ff=D_FF,
    use_repo=USE_REPO,
    use_flash=USE_FLASH,
)

model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters     : {total_params:,} ({total_params/1e6:.1f}M)")
print(f"Trainable parameters : {trainable_params:,}")
print(f"REPO-Attention       : {'ON' if USE_REPO else 'OFF'}")
print(f"Flash-Attention      : {'ON' if USE_FLASH else 'OFF'}")

if torch.cuda.is_available():
    print(f"GPU                  : {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory           : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 5. Sanity Check (forward + backward with dummy data)

In [ ]:
print("Running sanity check...")
model.train()

dummy_ids = torch.randint(0, VOCAB_SIZE, (2, MAX_SEQ_LEN), device=device)
dummy_labels = torch.randint(0, VOCAB_SIZE, (2, MAX_SEQ_LEN), device=device)

# Forward (same logic as your train.py)
embeddings = model.tgt_embed(dummy_ids)
output = embeddings
for layer in model.decoder.layers:
    output, _ = layer(output, tgt_mask=None, use_cache=False)
output = model.decoder.norm(output)
logits = model.project(output)

# Loss (same as your train.py)
shift_logits = logits[..., :-1, :].contiguous()
shift_labels = dummy_labels[..., 1:].contiguous()
loss = nn.CrossEntropyLoss(ignore_index=-100)(
    shift_logits.view(-1, shift_logits.size(-1)),
    shift_labels.view(-1)
)

# Backward
loss.backward()

print(f"[PASS] logits shape : {logits.shape}")
print(f"[PASS] loss         : {loss.item():.4f}")
print(f"[PASS] loss finite  : {torch.isfinite(loss).item()}")
print(f"[PASS] grads OK     : {all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters() if p.requires_grad)}")

model.zero_grad(set_to_none=True)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Sanity check passed!")

## 6. Create Dataset & DataLoader (using your `SlimPajamaDataset`)

In [ ]:
train_dataset = SlimPajamaDataset(
    data_dir=DATASET_DIR,
    tokenizer=tokenizer,
    max_length=MAX_SEQ_LEN,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    num_workers=4,       # adjust based on your Azure VM CPU cores
    pin_memory=True,
    prefetch_factor=2,
)

print(f"DataLoader ready (batch_size={BATCH_SIZE}, num_workers=4)")

## 7. WandB Init

In [ ]:
wandb.login(key="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")

if USE_WANDB:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_name = f"slimpajama_pretrain_{timestamp}"
    
    wandb.init(
        project=WANDB_PROJECT,
        name=run_name,
        config={
            "model_type": "decoder_only_transformer",
            "d_model": D_MODEL,
            "num_layers": NUM_LAYERS,
            "num_heads": NUM_HEADS,
            "num_kv_heads": KV_HEADS,
            "d_ff": D_FF,
            "vocab_size": VOCAB_SIZE,
            "max_sequence_length": MAX_SEQ_LEN,
            "dropout": DROPOUT,
            "learning_rate": LEARNING_RATE,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "weight_decay": WEIGHT_DECAY,
            "gradient_clipping": MAX_GRAD_NORM,
            "warmup_steps": WARMUP_STEPS,
            "dataset": "SlimPajama-6B",
            "dataset_source": "oxen.ai",
            "mixed_precision": True,
            "device": str(device),
            "architecture_features": ["GQA", "REPO-Attention", "Flash-Attention", "RMSNorm"],
            "total_params": total_params,
        },
        tags=["pre-training", "slimpajama", "gqa", "repo-attention", "flash-attention"]
    )
    wandb.watch(model, log="all", log_freq=200)
    print(f"WandB run started: {wandb.run.url}")
else:
    print("WandB disabled.")

## 8. Training Loop

Uses the same training logic from your `train/train.py`:
- Mixed precision via `autocast` + `GradScaler`
- Same forward pass: `tgt_embed` -> decoder layers -> norm -> project
- Same loss: `CrossEntropyLoss(ignore_index=pad_token_id)`
- Same checkpoint saving via your `save_checkpoint()`
- Auto-checkpoint every 2 hours

In [ ]:
# ======================== TRAINING ========================

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY, betas=(0.9, 0.98)
)
scaler = GradScaler()

model.train()
global_step = 0
best_loss = float('inf')
last_checkpoint_time = time.time()
epoch_losses = []

print(f"Starting training...")
print(f"  Epochs         : {EPOCHS}")
print(f"  Batch size     : {BATCH_SIZE}")
print(f"  Learning rate  : {LEARNING_RATE}")
print(f"  Warmup steps   : {WARMUP_STEPS}")
print(f"  Mixed precision: True")
print()

for epoch in range(EPOCHS):
    total_loss = 0
    epoch_start_time = time.time()
    batch_count = 0

    for i, batch in enumerate(train_loader):
        current_time = time.time()

        # ---- Auto-save every 2 hours (same as your train.py) ----
        if current_time - last_checkpoint_time >= 7200:
            print(f"\nAuto-saving checkpoint at epoch {epoch+1}, batch {i}...")
            avg_loss = total_loss / max(i, 1)
            save_checkpoint(
                model, optimizer, epoch, global_step, avg_loss, best_loss,
                CHECKPOINT_DIR, f"auto_checkpoint_epoch_{epoch+1}_step_{global_step}.pt"
            )
            last_checkpoint_time = current_time

        # ---- Get batch ----
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        # ---- Forward pass (same as your train.py) ----
        with autocast(device_type=device.type):
            embeddings = model.tgt_embed(input_ids)
            output = embeddings
            for layer in model.decoder.layers:
                output, _ = layer(output, tgt_mask=None, use_cache=False)
            output = model.decoder.norm(output)
            logits = model.project(output)

            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()

            loss = nn.CrossEntropyLoss(ignore_index=-100)(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1)
            )

        # ---- Backward + step (same as your train.py) ----
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        global_step += 1
        batch_count += 1
        epoch_losses.append(loss.item())

        # ---- WandB logging (same as your train.py) ----
        if USE_WANDB:
            log_dict = {
                "train/loss": loss.item(),
                "train/epoch": epoch + 1,
                "train/global_step": global_step,
                "train/learning_rate": optimizer.param_groups[0]['lr'],
            }
            if torch.cuda.is_available():
                log_dict.update({
                    "system/gpu_memory_allocated_gb": torch.cuda.memory_allocated() / 1e9,
                    "system/gpu_memory_reserved_gb": torch.cuda.memory_reserved() / 1e9,
                })
            wandb.log(log_dict, step=global_step)

        # ---- Print progress (same as your train.py) ----
        if i % 5 == 0:
            elapsed_time = time.time() - epoch_start_time
            gpu_memory = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
            print(f"Epoch {epoch+1}, Batch {i}, Loss: {loss.item():.4f}, "
                  f"Time: {elapsed_time:.1f}s, Step: {global_step}, GPU: {gpu_memory:.1f}GB")

        if i % 10 == 0:
            torch.cuda.empty_cache()

    # ---- End of epoch (same as your train.py) ----
    avg_loss = total_loss / max(batch_count, 1)
    epoch_duration = time.time() - epoch_start_time
    print(f"Epoch {epoch+1}/{EPOCHS}, Avg Loss: {avg_loss:.4f}, Duration: {epoch_duration:.1f}s")

    if USE_WANDB:
        wandb.log({
            "epoch/avg_loss": avg_loss,
            "epoch/duration_seconds": epoch_duration,
            "epoch/batches_processed": batch_count,
            "epoch/min_loss": min(epoch_losses[-batch_count:]) if batch_count > 0 else 0,
            "epoch/max_loss": max(epoch_losses[-batch_count:]) if batch_count > 0 else 0,
        }, step=global_step)

    if avg_loss < best_loss:
        best_loss = avg_loss
        print(f"New best loss: {best_loss:.4f} - Saving best model...")
        save_checkpoint(
            model, optimizer, epoch, global_step, avg_loss, best_loss,
            CHECKPOINT_DIR, "best_model.pt"
        )
        if USE_WANDB:
            wandb.log({"train/best_loss": best_loss}, step=global_step)

    save_checkpoint(
        model, optimizer, epoch, global_step, avg_loss, best_loss,
        CHECKPOINT_DIR, f"epoch_{epoch+1}_checkpoint.pt"
    )

# ---- Final save ----
print("Training completed! Saving final checkpoint...")
save_checkpoint(
    model, optimizer, EPOCHS - 1, global_step, avg_loss, best_loss,
    CHECKPOINT_DIR, "final_model.pt"
)

if USE_WANDB:
    wandb.finish()

print("Done!")

## 9. Resume from Checkpoint

In [ ]:
# Uncomment to resume training from a checkpoint
"""
RESUME_PATH = os.path.join(CHECKPOINT_DIR, "best_model.pt")

if os.path.exists(RESUME_PATH):
    checkpoint = torch.load(RESUME_PATH, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    global_step = checkpoint['global_step']
    best_loss = checkpoint['best_loss']
    start_epoch = checkpoint['epoch'] + 1
    print(f"Resumed from {RESUME_PATH} at step {global_step}, epoch {start_epoch}")
else:
    print(f"No checkpoint found at {RESUME_PATH}")
"""
print("Resume cell ready (uncomment to use).")